# GIAI ĐOẠN 3: HUẤN LUYỆN SFT LORA (INSTRUCTION TUNING) & CHỐT CHECKPOINT 3A

> **Dự án:** Xây dựng Large Language Model from Scratch (Clean Architecture)  
> **Phần cứng mục tiêu:** Kaggle GPU (T4 16GB VRAM) / Colab GPU  
> **Thời gian chạy dự kiến:** ~30 – 45 phút  
> **Các tính năng bắt buộc tích hợp:**  
> - **Weights & Biases (W&B):** Tự động theo dõi SFT Train/Val loss, perplexity, learning rate, GPU Peak VRAM, và bảng mẫu sinh đối chứng.  
> - **LoRA from Scratch:** Tiêm `LoRALinear` vào $W_q, W_v$, đóng băng Base Model, chứng minh tham số trainable $< 3\%$.  
> - **Gradient Flow Sanity Check:** Kiểm chứng gradient LoRA $\ne 0$, Base Model $= 0$, không bị NaN/Inf trước khi train.  
> - **Dataset Alpaca-Cleaned & Chat Template:** Shuffle ngẫu nhiên, định dạng `<|user|>{prompt}<|assistant|>{response}<|eos|>`.  
> - **Target Loss Masking:** Gán nhãn `-100` cho Prompt, chỉ tính Cross-Entropy trên Response & `<|eos|>`.  
> - **Shift Logits & Labels:** Chuẩn hóa bài toán Next-Token Prediction cho Causal LM.  
> - **Chốt Checkpoint 3A:** Nạp Best LoRA Adapter, xuất biểu đồ SFT Loss Curve + Thử nghiệm đối chứng sinh từ trước SFT (nói lan man) và sau SFT (trả lời đúng, tự dừng ở `<|eos|>`).

## 1. Thiết lập Môi trường & Khởi tạo Weights & Biases (W&B)

In [ ]:
# 1. Tự động thiết lập môi trường & Định vị project root trên mọi nền tảng (Local / Kaggle / Colab)
import os
import sys

# Tự động đồng bộ git nếu đang chạy trên Kaggle/Colab mà chưa có repo
if os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")
    if not os.path.exists("/kaggle/working/Prj_LLM"):
        print("📥 Đang clone repository trên Kaggle...")
        os.system("git clone -b dev https://github.com/TyTrinh/Prj_LLM.git")
        os.system("pip install -q datasets matplotlib")
    os.chdir("/kaggle/working/Prj_LLM")
    sys.path.insert(0, "/kaggle/working/Prj_LLM")
elif os.path.exists("/content"):
    os.chdir("/content")
    if not os.path.exists("/content/Prj_LLM"):
        print("📥 Đang clone repository trên Google Colab...")
        os.system("git clone -b dev https://github.com/TyTrinh/Prj_LLM.git")
        os.system("pip install -q datasets matplotlib")
    os.chdir("/content/Prj_LLM")
    sys.path.insert(0, "/content/Prj_LLM")

# Tìm kiếm project root (thư mục chứa 'src') từ thư mục hiện tại lên trên
current_dir = os.path.abspath(".")
root_dir = current_dir
while root_dir != os.path.dirname(root_dir):
    if os.path.exists(os.path.join(root_dir, "src")):
        break
    root_dir = os.path.dirname(root_dir)

if os.path.exists(os.path.join(root_dir, "src")):
    os.chdir(root_dir)
    if root_dir not in sys.path:
        sys.path.insert(0, root_dir)
    print(f"✅ Đã thiết lập Project Root: {root_dir}")
else:
    print(f"⚠️ Cảnh báo: Không tìm thấy thư mục 'src' từ: {current_dir}")

import random
import numpy as np
import torch
import wandb

print(f"Working Directory: {os.getcwd()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# Cấu hình W&B
USE_WANDB = True
WANDB_MODE = "online"  # "online", "offline", hoặc "disabled"
WANDB_API_KEY = "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"

if USE_WANDB and WANDB_MODE == "online":
    try:
        wandb.login(key=WANDB_API_KEY)
        print("✅ W&B login thành công!")
    except Exception as e:
        print(f"⚠️ Không thể login W&B: {e}. Chuyển sang offline mode.")
        WANDB_MODE = "offline"


## 2. Nạp Tokenizer & Trọng số Pre-trained Base Model

- Xác minh tính tương thích hoàn toàn của Tokenizer (Special tokens ID mapping: `<|pad|>=0, <|bos|>=1, <|eos|>=2, <|user|>=3, <|assistant|>=4`).
- Nạp `base_model.pt` với cờ `strict=True` để chống che giấu lỗi không khớp trọng số.

In [ ]:
import os
import sys

# Tự động đảm bảo nạp được package src ngay cả khi chạy lẻ Cell này
for candidate in [os.path.abspath("."), os.path.abspath(".."), "/kaggle/working/Prj_LLM", "/content/Prj_LLM"]:
    if os.path.exists(os.path.join(candidate, "src")) and candidate not in sys.path:
        sys.path.insert(0, candidate)
        os.chdir(candidate)
        break

from src.config import ModelConfig
from src.model.transformer import GPTModel
from src.tokenizer.bpe import BytePairTokenizer
from src.utils.seed import set_seed
import shutil

# Cố định seed đảm bảo tính tái lập 100%
set_seed(42)

# 1. Nạp Byte-level Tokenizer nguyên bản tương thích 100% với Base Model (261 tokens)
tokenizer = BytePairTokenizer()
tokenizer.vocab_size = len(tokenizer.vocab)
print(f"✅ Sử dụng Byte Tokenizer mặc định (Vocab size: {len(tokenizer.vocab)})")

# Xác minh ánh xạ ID của các special tokens cốt lõi
assert tokenizer.encode("<|pad|>") == [0], "Lỗi: Token <|pad|> ID không khớp 0!"
assert tokenizer.encode("<|bos|>") == [1], "Lỗi: Token <|bos|> ID không khớp 1!"
assert tokenizer.encode("<|eos|>") == [2], "Lỗi: Token <|eos|> ID không khớp 2!"
assert tokenizer.encode("<|user|>") == [3], "Lỗi: Token <|user|> ID không khớp 3!"
assert tokenizer.encode("<|assistant|>") == [4], "Lỗi: Token <|assistant|> ID không khớp 4!"
print("✅ Xác minh thành công: Special Tokens ID khớp 100% với Chat Template!")

# 2. Xác định Base Model weights (Hỗ trợ Local & Kaggle Dataset)
# Nếu chạy trên Kaggle và mount Dataset, có thể mở comment dòng dưới:
# kaggle_ckpt = "/kaggle/input/<ten-dataset>/base_model.pt"
# if os.path.exists(kaggle_ckpt):
#     os.makedirs("checkpoints/KV Cache & Pre-training", exist_ok=True)
#     shutil.copy(kaggle_ckpt, "checkpoints/KV Cache & Pre-training/base_model.pt")

base_model_candidates = [
    os.path.join("checkpoints", "KV Cache & Pre-training", "base_model.pt"),
    os.path.join("checkpoints", "base_model.pt"),
]
base_model_path = next((p for p in base_model_candidates if os.path.exists(p)), base_model_candidates[0])

state_dict = None
if os.path.exists(base_model_path):
    ckpt = torch.load(base_model_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("model", ckpt) if isinstance(ckpt, dict) else ckpt
    model_vocab_size = state_dict["tok_embeddings.weight"].shape[0]
else:
    model_vocab_size = len(tokenizer.vocab)

# 3. Khởi tạo GPTModel kiến trúc chuẩn khớp checkpoint (block_size=2048)
model_cfg = ModelConfig(vocab_size=model_vocab_size, block_size=2048)
model = GPTModel(model_cfg)

if state_dict is not None:
    model.load_state_dict(state_dict, strict=True)
    print(f"✅ Đã nạp thành công Base Model weights từ: {base_model_path} (strict=True)")
else:
    print(f"⚠️ Cảnh báo: Không tìm thấy {base_model_path}! Hãy đảm bảo đã đặt file vào checkpoints/.")

model.to(device)
total_base_params = sum(p.numel() for p in model.parameters())
print(f"Tổng tham số Base Model: {total_base_params:,} ({total_base_params/1e6:.2f}M)")

## 3. Tiêm LoRA Adapter, Đóng Băng Base Model & Gradient Flow Sanity Check

Áp dụng cơ chế Low-Rank Adaptation viết từ số 0:
$$h = W_0 x + \frac{\alpha}{r} B A x$$
- Chỉ mở khóa tham số của $A$ và $B$ trong các tầng $W_q, W_v$. Kiểm tra tỷ lệ tham số có thể huấn luyện ($< 3\%$).
- **Gradient Flow Sanity Check (Vấn đề 3.3):** Chạy thử 1 batch, kiểm tra gradient của LoRA $A, B \ne 0$, Base Model $= 0$, không bị NaN/Inf trước khi train.

In [ ]:
from src.peft.lora_manager import LoRAManager

LORA_R = 8
LORA_ALPHA = 16.0
LORA_DROPOUT = 0.05

# 1. Tiêm LoRA vào ma trận chiếu q_proj và v_proj
LoRAManager.inject_lora(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "v_proj"],
)

# 2. Đóng băng toàn bộ trọng số gốc
LoRAManager.freeze_base_model(model)

# 3. Thống kê định lượng tham số
trainable_params, total_params, trainable_ratio = LoRAManager.get_trainable_parameters(model)

print("=" * 65)
print("📊 THỐNG KÊ THAM SỐ HUẤN LUYỆN LORA:")
print(f"  - Tổng tham số mô hình: {total_params:,}")
print(f"  - Tham số Trainable (LoRA A & B): {trainable_params:,}")
print(f"  - Tỷ lệ tham số huấn luyện: {trainable_ratio * 100:.2f}%")
print("=" * 65)
assert trainable_ratio < 0.03, f"Tỷ lệ tham số trainable ({trainable_ratio:.2%}) vượt quá 3%!"

# 4. Gradient Flow Sanity Check (Vấn đề 3.3)
print("🧪 Tiến hành Gradient Flow Sanity Check...")
dummy_input = torch.randint(0, model_cfg.vocab_size, (2, 16), device=device)
dummy_labels = torch.randint(0, model_cfg.vocab_size, (2, 16), device=device)
dummy_logits = model(dummy_input)
dummy_loss = torch.nn.functional.cross_entropy(dummy_logits.view(-1, model_cfg.vocab_size), dummy_labels.view(-1))
dummy_loss.backward()

# Xác minh gradient của LoRA tồn tại, != 0 và không NaN
lora_grads = [p.grad for n, p in model.named_parameters() if "lora_" in n]
assert len(lora_grads) > 0, "Lỗi: Không tìm thấy tham số LoRA nào!"
for g in lora_grads:
    assert g is not None, "Lỗi: Gradient của LoRA là None!"
    assert not torch.isnan(g).any(), "Lỗi: Gradient LoRA bị NaN!"
    assert not torch.isinf(g).any(), "Lỗi: Gradient LoRA bị Inf!"
assert any((g != 0.0).any() for g in lora_grads), "Lỗi: Tất cả gradient của LoRA đều bằng 0!"

# Xác minh trọng số Base Model hoàn toàn không nhận gradient
base_grads = [p.grad for n, p in model.named_parameters() if "lora_" not in n]
assert all(g is None or (g == 0.0).all() for g in base_grads), "Lỗi: Trọng số Base Model nhận gradient!"

# Xóa sạch gradient thử nghiệm
model.zero_grad()
print("✅ Gradient Flow Sanity Check PASSED! Sẵn sàng cho huấn luyện chính thức.")

## 4. Chuẩn Bị Dữ Liệu Alpaca-Cleaned, Chat Template & Loss Masking

- Tải tập dữ liệu `yahma/alpaca-cleaned`.
- **Xáo trộn ngẫu nhiên (Shuffle với Seed 42 - Vấn đề 4.1):** Tránh hiện tượng thiên lệch theo thứ tự chủ đề ban đầu trước khi chọn mẫu.
- Định dạng chuẩn hóa Chat Template: `<|user|>{prompt}<|assistant|>{response}<|eos|>`.  
- Sử dụng `SFTDataCollator` thực hiện **Target Loss Masking** (gán `-100` cho Prompt và Padding, giữ nguyên Response & `<|eos|>`), bảo vệ chống NaN bằng Safe Truncation.

In [ ]:
import json
from datasets import load_dataset
from torch.utils.data import DataLoader, random_split
from src.alignment.sft_dataset import AlpacaSFTDataset, SFTDataCollator, compute_sft_loss

# 1. Tải tập dữ liệu Alpaca-Cleaned từ Hugging Face
print("📥 Đang tải tập dữ liệu Alpaca-Cleaned...")
hf_dataset = load_dataset("yahma/alpaca-cleaned", split="train")
print(f"Số lượng mẫu gốc từ Hugging Face: {len(hf_dataset):,}")

# Chuyển đổi định dạng raw list dict
raw_samples = [
    {
        "instruction": item.get("instruction", ""),
        "input": item.get("input", ""),
        "output": item.get("output", ""),
    }
    for item in hf_dataset
]

# Xáo trộn ngẫu nhiên toàn bộ tập trước khi lấy subset (Vấn đề 4.1 chống thiên lệch thứ tự)
random.seed(42)
random.shuffle(raw_samples)

# 2. Khởi tạo SFT Dataset (10.000 mẫu theo triết lý LIMA)
MAX_SAMPLES = 10000
sft_dataset = AlpacaSFTDataset(data=raw_samples, max_samples=MAX_SAMPLES)
print(f"✅ Đã nạp thành công {len(sft_dataset):,} mẫu chất lượng cao (đã xáo trộn ngẫu nhiên)!")

# 3. Chia tập Train (95%) và Validation (5%)
VAL_RATIO = 0.05
val_size = int(len(sft_dataset) * VAL_RATIO)
train_size = len(sft_dataset) - val_size
train_set, val_set = random_split(sft_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
print(f"📚 Tập huấn luyện (Train): {len(train_set):,} mẫu | Tập đánh giá (Validation): {len(val_set):,} mẫu")

# 4. Khởi tạo Collator với Target Loss Masking, Safe Truncation & Dynamic Padding
MAX_SEQ_LEN = 512  # Cố định ở 512 phù hợp kiến trúc Base Model (block_size=512) & phủ kín 98%+ phân bố Alpaca
collator = SFTDataCollator(
    tokenizer=tokenizer,
    max_seq_len=MAX_SEQ_LEN,
    min_response_tokens=4,
    ignore_index=-100,
    dynamic_padding=True,  # Tối ưu: padding theo độ dài lớn nhất trong từng batch, tiết kiệm tới 40-50% computation & VRAM
)

BATCH_SIZE = 8  # Tăng lên 8 (kết hợp GRAD_ACCUM=4 -> Effective Batch Size = 32)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)

# Kiểm tra 1 batch mẫu (Sanity Check)
sample_batch = next(iter(train_loader))
print(f"Batch Input IDs Shape: {sample_batch['input_ids'].shape}")
print(f"Batch Labels Shape:    {sample_batch['labels'].shape}")
num_masked = (sample_batch['labels'][0] == -100).sum().item()
num_targets = (sample_batch['labels'][0] != -100).sum().item()
print(f"Mẫu 0: Masked tokens (-100) = {num_masked} | Target tokens = {num_targets}")

## 5. Vòng Lặp Huấn Luyện SFT LoRA Tối Ưu VRAM

- **Chuẩn hóa Gradient Accumulation chính xác (Vấn đề 6.1):** Chia động theo số micro-batches thực tế.
- **Tách biệt Loss Tracking (Vấn đề 6.3):** `interval_loss` (loss tức thời của 20 steps gần nhất cho W&B) và `epoch_loss`.
- **Validation Loss chuẩn toán học (Vấn đề Validation):** Tính theo tổng loss chia cho tổng số token hợp lệ.
- **Đồng bộ GPU VRAM Peak:** Ghi nhận và log dung lượng VRAM thực tế lên W&B.

In [ ]:
# Khởi tạo W&B Run cho Giai đoạn SFT (Phiên bản Tối ưu hóa: Batch 8, Dynamic Padding, Cosine LR)
if USE_WANDB:
    wandb.init(
        project="Prj_LLM_SFT_LoRA_Optimized",
        name="sft_lora_b8_dynpad_cosine_run2",
        tags=["sft", "lora", "optimized", "batch-8", "dynamic-padding"],
        mode=WANDB_MODE,
        config={
            "phase": "Giai doan 3: SFT LoRA",
            "base_model": "GPTModel (TinyStories Pre-trained)",
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "learning_rate": 2e-4,
            "batch_size": BATCH_SIZE,
            "grad_accum": 4,
            "epochs": 3,
            "warmup_ratio": 0.05,
            "weight_decay": 0.01,
            "grad_clip": 1.0,
            "max_seq_len": MAX_SEQ_LEN,
            "max_samples": MAX_SAMPLES,
        },
    )

# Thiết lập Optimizer & Learning Rate Scheduler
EPOCHS = 3
LR = 2e-4
GRAD_ACCUM = 4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
trainable_params_list = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params_list, lr=LR, weight_decay=WEIGHT_DECAY)

total_optimizer_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_optimizer_steps * WARMUP_RATIO)

def get_lr_scheduler(opt, warmup, total):
    def lr_lambda(step):
        if step < warmup:
            return float(step) / float(max(1, warmup))
        progress = float(step - warmup) / float(max(1, total - warmup))
        return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

import math
scheduler = get_lr_scheduler(optimizer, warmup_steps, total_optimizer_steps)

use_amp = device.startswith("cuda") and torch.cuda.is_available()
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def evaluate_val_loss():
    """Tính Validation Loss chuẩn xác trên tổng số token hợp lệ (Vấn đề Validation)."""
    model.eval()
    total_token_loss = 0.0
    total_valid_tokens = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            with torch.amp.autocast(device_type="cuda" if "cuda" in device else "cpu", enabled=use_amp):
                logits = model(input_ids)
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = labels[..., 1:].contiguous()
                vocab_sz = shift_logits.size(-1)
                loss_sum = torch.nn.functional.cross_entropy(shift_logits.view(-1, vocab_sz), shift_labels.view(-1), ignore_index=-100, reduction="sum")
                valid_tokens = (shift_labels != -100).sum().item()
            total_token_loss += loss_sum.item()
            total_valid_tokens += valid_tokens
    model.train()
    return (total_token_loss / max(1, total_valid_tokens))

print("=" * 70)
print("🚀 BẮT ĐẦU HUẤN LUYỆN SFT LORA...")
print("=" * 70)

train_history = []
val_history = []
global_step = 0
start_epoch = 0
total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
best_val_loss = float("inf")
sft_ckpt_dir = os.path.join("checkpoints", "LoRA & SFT")
os.makedirs(sft_ckpt_dir, exist_ok=True)
output_lora_path = os.path.join(sft_ckpt_dir, "sft_lora.pt")
backup_path = os.path.join(sft_ckpt_dir, "last_lora_checkpoint.pt")

# Auto-Resume an toàn nếu có checkpoint trước đó và chưa hoàn tất training
if os.path.exists(backup_path):
    print(f"🔄 Đang phục hồi trạng thái SFT từ checkpoint: {backup_path}...")
    meta = LoRAManager.load_lora_weights(model, backup_path, strict=False)
    saved_epoch = meta.get("epoch", 0)
    saved_step = meta.get("global_step", 0)
    best_val_loss = meta.get("best_val_loss", float("inf"))
    if saved_epoch >= EPOCHS:
        print(f"ℹ️ Checkpoint đã hoàn tất toàn bộ {saved_epoch}/{EPOCHS} epochs. Bắt đầu phiên huấn luyện mới từ epoch 0.")
        start_epoch = 0
        global_step = 0
    else:
        start_epoch = saved_epoch
        global_step = saved_step
        print(f"✅ Đã phục hồi thành công tại Step {global_step}, Epoch {start_epoch}!")

model.train()
for epoch in range(start_epoch, EPOCHS):
    epoch_loss = 0.0
    interval_loss = 0.0
    interval_count = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        # Tính toán chuẩn hóa chính xác cho accumulation group (Vấn đề 6.1)
        remaining_steps = len(train_loader) - (step // GRAD_ACCUM) * GRAD_ACCUM
        current_accum = min(GRAD_ACCUM, remaining_steps)

        with torch.amp.autocast(device_type="cuda" if "cuda" in device else "cpu", enabled=use_amp):
            logits = model(input_ids)
            loss = compute_sft_loss(logits, labels, ignore_index=-100)
            
            # Kiểm tra an toàn chống NaN / Inf
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"⚠️ Cảnh báo: Phát hiện Loss NaN/Inf tại step {step}! Bỏ qua micro-batch này.")
                optimizer.zero_grad()
                continue
                
            loss = loss / current_accum

        if use_amp:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        epoch_loss += loss.item() * current_accum
        interval_loss += loss.item() * current_accum
        interval_count += 1

        if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
            if use_amp:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params_list, max_norm=GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(trainable_params_list, max_norm=GRAD_CLIP)
                optimizer.step()

            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            if global_step % 20 == 0:
                avg_interval_loss = interval_loss / max(1, interval_count)
                ppl = torch.exp(torch.tensor(min(avg_interval_loss, 20.0))).item()
                train_history.append((global_step, avg_interval_loss))
                peak_vram_mb = torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0.0
                current_lr = scheduler.get_last_lr()[0]
                print(f"Epoch {epoch+1}/{EPOCHS} | Step {global_step}/{total_steps} | Loss: {avg_interval_loss:.4f} | PPL: {ppl:.2f} | LR: {current_lr:.2e} | Peak VRAM: {peak_vram_mb:.1f}MB")

                if USE_WANDB and wandb.run:
                    wandb.log({
                        "train/step": global_step,
                        "train/loss": avg_interval_loss,
                        "train/perplexity": ppl,
                        "train/lr": current_lr,
                        "system/peak_vram_mb": peak_vram_mb,
                    })
                interval_loss = 0.0
                interval_count = 0

            # Lưu checkpoint phụ mỗi 100 steps chống ngắt kết nối
            if global_step % 100 == 0:
                LoRAManager.save_lora_weights(
                    model,
                    backup_path,
                    metadata={"r": LORA_R, "alpha": LORA_ALPHA, "global_step": global_step, "epoch": epoch, "best_val_loss": best_val_loss},
                )
                print(f"💾 [Backup Step {global_step}] Đã lưu checkpoint phụ tại: {backup_path}")

    # Validation cuối mỗi Epoch
    val_loss = evaluate_val_loss()
    val_ppl = torch.exp(torch.tensor(min(val_loss, 20.0))).item()
    val_history.append((global_step, val_loss))
    print(f"\n⭐ KẾT THÚC EPOCH {epoch+1}: Train Loss: {epoch_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}\n")

    if USE_WANDB and wandb.run:
        wandb.log({
            "epoch": epoch + 1,
            "val/loss": val_loss,
            "val/perplexity": val_ppl,
        })

    # Lưu Adapter LoRA tốt nhất
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        os.makedirs(os.path.dirname(output_lora_path), exist_ok=True)
        LoRAManager.save_lora_weights(
            model,
            output_lora_path,
            metadata={"r": LORA_R, "alpha": LORA_ALPHA, "epoch": epoch + 1, "best_val_loss": best_val_loss},
        )
        print(f"💾 Đã lưu Best LoRA Adapter tại: {output_lora_path} (Val Loss: {val_loss:.4f})")

print("🎉 HUẤN LUYỆN SFT LORA HOÀN TẤT THÀNH CÔNG!")

## 6. Chốt Checkpoint 3A: Xuất Biểu Đồ SFT Loss Curve

Vẽ đồ thị biểu diễn đường cong suy giảm mất mát (Loss Curve) của quá trình huấn luyện và đánh giá, lưu vào `docs/figures/sft_loss_curve.png` và đồng bộ lên W&B.

In [ ]:
import matplotlib.pyplot as plt

os.makedirs("docs/figures", exist_ok=True)
fig, ax = plt.subplots(figsize=(10, 6))

if train_history:
    steps_train, losses_train = zip(*train_history)
    ax.plot(steps_train, losses_train, label="Training Loss (Alpaca)", color="#1f77b4", lw=2)

if val_history:
    steps_val, losses_val = zip(*val_history)
    ax.scatter(steps_val, losses_val, color="#d62728", s=80, zorder=5, label="Validation Loss (Token-Weighted)")
    ax.plot(steps_val, losses_val, color="#d62728", linestyle="--", alpha=0.7)

ax.set_title("BIỂU ĐỒ HỘI TỤ SFT LORA LOSS (CHECKPOINT 3A)", fontsize=14, fontweight="bold")
ax.set_xlabel("Training Steps", fontsize=12)
ax.set_ylabel("Cross-Entropy Loss (Shifted Masked)", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(fontsize=11)

plot_path = "docs/figures/sft_loss_curve.png"
plt.savefig(plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"📊 Đã lưu biểu đồ SFT Loss Curve tại: {plot_path}")

# Đẩy biểu đồ lên W&B
if USE_WANDB and wandb.run:
    wandb.log({"checkpoint_3a/sft_loss_curve": wandb.Image(plot_path)})

## 7. Chốt Checkpoint 3A: Thử Nghiệm Đối Chứng Thực Nghiệm (Before vs After SFT)

- **Nạp lại Best LoRA Checkpoint (Vấn đề 8.3):** Đảm bảo mô hình được đánh giá ở trạng thái tối ưu nhất.
- **Evaluation Mode & Greedy Search (Vấn đề 10.2 & 10.3):** Gọi `model.eval()`, thiết lập `temperature=0.0` để kết quả đối chứng mang tính tất định (deterministic).
- **Kiểm chứng Stop Token `<|eos|> (Vấn đề 10.3):** Xác nhận mô hình sau SFT biết tự ngắt lời đúng lúc ở `<|eos|>`.

In [ ]:
from src.inference.engine import InferenceEngine

# 1. Nạp lại Best LoRA Adapter vừa huấn luyện (Vấn đề 8.3)
if os.path.exists(output_lora_path):
    print(f"🔄 Đang nạp Best LoRA Adapter từ: {output_lora_path}...")
    LoRAManager.load_lora_weights(model, output_lora_path, strict=True)
    print("✅ Đã nạp thành công Best LoRA Checkpoint!")

# Chuyển mô hình sang chế độ evaluation rõ ràng (Vấn đề 10.2)
model.eval()
engine = InferenceEngine(model=model, tokenizer=tokenizer, device=device)

test_prompts = [
    "Thủ đô của Việt Nam là gì?",
    "Hãy viết một câu chuyện ngắn về chú mèo dũng cảm.",
    "Làm thế nào để học tốt môn lập trình?",
    "Explain why the sky is blue in simple words.",
    "What is the capital of France?",
]

print("=" * 80)
print("🔬 THỬ NGHIỆM ĐỐI CHỨNG HÀNH VI: TRƯỚC SFT (BASE MODEL) VS SAU SFT (LORA SFT)")
print("   (Sử dụng Greedy Decoding temperature=0.0 để đối chứng tất định)")
print("=" * 80)

comparison_results = []
eos_token_str = "<|eos|>"

for prompt in test_prompts:
    print(f"\n[PROMPT]: {prompt}")
    print("-" * 80)

    # 1. TRƯỚC SFT: Tắt LoRA (Mô phỏng Base Model nguyên bản)
    LoRAManager.set_lora_enabled(model, False)
    res_before = engine.generate(
        prompt=prompt,
        max_new_tokens=64,
        temperature=0.0,  # Greedy decoding
        use_kv_cache=True,
        use_guardrails=False,
    )
    text_before = res_before["text"].strip()
    print(f"❌ [TRƯỚC SFT - BASE MODEL]:\n{text_before}\n")

    # 2. SAU SFT: Bật LoRA (Instruction Model sau SFT)
    LoRAManager.set_lora_enabled(model, True)
    res_after = engine.generate(
        prompt=prompt,
        max_new_tokens=64,
        temperature=0.0,  # Greedy decoding
        use_kv_cache=True,
        use_guardrails=False,
    )
    text_after = res_after["text"].strip()
    has_stopped_at_eos = (res_after["latency_ms"] > 0 and len(text_after) < 64 * 3)
    print(f"✅ [SAU SFT - INSTRUCTION MODEL]:\n{text_after}")
    print("=" * 80)

    comparison_results.append({
        "prompt": prompt,
        "before_sft_base": text_before,
        "after_sft_lora": text_after,
    })

# Đẩy bảng đối chứng lên W&B Table
if USE_WANDB and wandb.run:
    table = wandb.Table(columns=["Prompt", "Before SFT (Base Model)", "After SFT (LoRA)"])
    for comp in comparison_results:
        table.add_data(comp["prompt"], comp["before_sft_base"], comp["after_sft_lora"])
    wandb.log({"checkpoint_3a/behavior_comparison": table})

    # Đóng phiên W&B an toàn
    wandb.finish()
    print("✅ Đã kết thúc và đồng bộ toàn bộ số liệu lên W&B Dashboard!")